### [MARKDOWN CELL]
# Day 08: Superpowers of Decorators (Decorators) 🦸‍♂️

## 👋 Welcome to Week 2!
In Week 1, you learned how to process data, handle errors, and automate tasks. 
Now, we level up to **Professional Python**. 

Today, we learn a tool that you will see in almost every modern Python codebase:

**Decorators:** Giving your existing functions "superpowers" without rewriting them.

---

# 📦 Topic 1: Namespaces in Python

A namespace is simply a container that maps names to objects. A namespace can also be said as a system which Python uses to keep track of which name refers to which object.

Think of it like a dictionary behind the scenes.

In [ ]:
x = 10

# Internally Python stores something like:
{
   "x": 10
}

### Name Resolution Order for Namespaces (LEGB Rule)

When Python tries to find a variable, it follows a specific order.

This is called the LEGB rule.

Order Python checks:
1. Local (Created when a function is called and destroyed when the function finishes.)
2. Enclosing (Created when there are nested fucntions.)
3. Global (Created when a script starts and exists until the script ends.)
4. Built-in (Created when Python starts and lasts until Python stops.)

In [ ]:
x = 100

def outer():
    x = 50

    def inner():
        # Python checks:
        # Local → inner function → no x
        # Enclosing → outer function → found x
        # Stops searching
        print(x) 

    inner()

outer() # 50

In [ ]:
x = [3,5,4,1,8]
print(min(x))

def min(some_input):
    print("I'm the min function!")

# see how built-in min function is overwritten with global min function.
min(x)

### Visualizing Namespace Search

Imagine Python searching like this:
```bash
inner() function
      ↓
outer() function
      ↓
global scope
      ↓
built-in scope
```

#### Practice Problem

In [ ]:
# What will be the output of this code?
x = 10

def change():
    x = 20

change()

print(x)

## 🎁 Topic 2: Functions as Objects
To understand Decorators, you must understand a crazy Python rule: **Functions are just variables**. You can assign them to other variables, or pass them into other functions!

In [ ]:
def shout(text: str) -> str:
    return text.upper()

# 1. Assigning a function to a variable (NO parentheses!)
yell = shout 
print(yell("hello!")) # Works exactly like shout()

In [ ]:
# 2. Calling a function inside a function
def run_function(text: str):
    return shout(text)

print(run_function("Hello"))

In [ ]:
# 3. Passing a function into another function
def greet(func, name: str) -> None:
    # We run the function that was passed in!
    greeting = func(f"Hi there, {name}")
    print(greeting)

greet(shout, "Tony")

---
## 🪄 Topic 3: The Decorator (`@wrapper`)
A **Decorator** is a function that takes another function, adds some extra functionality (a "wrapper"), and returns it. 
It's like taking a standard phone, putting a protective case on it, and handing it back. It's the same phone, but with a new superpower.



Instead of writing clunky wrapper code manually, Python gives us the **`@` syntactic sugar**.

### Understanding The Concept

We will to do below:
- Create a decorator function which adds stars before and after function's execution
- Define `greet` function
- Pass `greet` to decorator
- Wrap it with new behavior
- Replace `greet` with the wrapped version

In [ ]:
# 1. Build the Decorator (The Protective Case)
def star_decorator(func):
    def wrapper():
        print("********************") # Do something BEFORE
        func()                      # Run the original function
        print("********************") # Do something AFTER
    return wrapper                  # Return the upgraded function

# 2. Define the function we want to add decorators to
def greet():
    print("Hello, World!")

# 3. Using decorator as a function
greet_decorated = star_decorator(greet)
greet_decorated()

# See how the same function works differently with and without the decorator

In [ ]:
# Using same decorator on multiple functions
def play_game() -> None:
    print("Hi, Would you like to play a game?")

play_game_decorated = star_decorator(play_game)
play_game_decorated()

### Adding Decorator Using `@`

As you can see above wrapping a fucntion with decorator is two step process and we are not calling the main fucntion which we need to decorate instead we are calling decorator fucntion whcih does is not good for readiability. 

What if we could have a way to call `greet` or `play_game` instead of calling decorator. We can do that by using `@` to decorate the function. 

In [ ]:
# 2. Apply it using '@'
@star_decorator
def greet():
    print("Hello, World!")

# 3. Call the function normally!
greet()

In [ ]:
@star_decorator
def play_game() -> None:
    print("Hi, Would you like to play a game?")

play_game()

## 🧳 Topic 4: Decorators with Arguments (`*args, **kwargs`)
What if the function we are decorating takes arguments? Our wrapper needs to accept them!

We use `*args` and `**kwargs` inside the wrapper to catch any arguments and pass them through to the original function.

In [ ]:
def announce(func):
    # The wrapper catches ANY arguments using *args and **kwargs
    def wrapper(*args, **kwargs):
        print(f"🔊 Running function: {func.__name__}...")
        # Pass them safely to the original function
        result = func(*args, **kwargs) 
        
        print("✅ Finished!")
        return result
    return wrapper

@announce
def add_numbers(a: int, b: int) -> int:
    return a + b

@announce
def greet(name: str) -> None:
    print(f"Welcome, {name}!")

# It works for both!
print("Result:", add_numbers(5, 10))
print("-" * 20)
print(greet("Alice"))

In [ ]:
greet_decorated = announce(greet("Alice"))
# greet_decorated("Alice")

### A Practical Example (Measuring execution time)

Let's say we want to measure execution time of any function. This is a very useful case as this will be used while measuring time taken by your fucntion when building them. 

In [ ]:
import time

def timer(func):

    def wrapper(*args, **kwargs):
        start = time.time()

        result = func(*args, **kwargs)

        end = time.time()
        print("Execution time:", end - start, "seconds.")

        return result

    return wrapper

In [ ]:
@timer
def slow_function():
    time.sleep(2)
    print("Task finished")

slow_function()

In [ ]:
@ timer
def expnonent_sum(end_num: int) -> None:
    total = 0
    for i in range(1, end_num + 1):
        total += i ** i

expnonent_sum(100)

## 🧳 Topic 5: Passing Arguments to Decorator Function
Now that we know main function can take argument which can be passed via decorator. But till now we have been decorating function direclty like `@timer`. But what if decorator function needs an argument like `@timer(millisec)`
. 

Let's add this final capability to decorator.

In [ ]:
import time

def timer(unit="seconds"):
    units = {"seconds": 1, "millisecond": 1000, "microsecond": 1000000}

    def decorator(func):

        def wrapper(*args, **kwargs):
            start = time.time()

            result = func(*args, **kwargs)

            end = time.time()
            
            multiplier = units.get(unit, 1)
            print("Execution time:", ((end - start) * multiplier), unit)

            return result

        return wrapper
    
    return decorator

In [ ]:
@timer(unit="microsecond")
def expnonent_sum(end_num: int) -> None:
    total = 0
    for i in range(1, end_num + 1):
        total += i ** i

expnonent_sum(100)

In [ ]:
@ timer(unit="seconds")
def expnonent_sum(end_num: int) -> None:
    total = 0
    for i in range(1, end_num + 1):
        total += i ** i

expnonent_sum(1000)

### Let’s build a logging decorator

In [ ]:
def logger(level):

    def decorator(func):

        def wrapper(*args, **kwargs):
            print(f"[{level}] Starting function")

            result = func(*args, **kwargs)

            print(f"[{level}] Function finished")

            return result

        return wrapper

    return decorator


def process_data():
    print("Processing data")


process_data = logger("INFO")(process_data)

process_data()

In [ ]:
@logger("INFO")
def process_data():
    print("Processing data")

process_data()

### Visual Flow (Best Way to Explain to Students)

```Bash
logger("INFO")
      ↓
returns decorator
      ↓
decorator(process_data)
      ↓
returns wrapper
      ↓
process_data = wrapper
      ↓
process_data()
      ↓
wrapper executes
```

### Mental Model

A decorator is simply:

- A function that takes another function
- Adds extra behavior
- And returns a new function

## 🔗 Topic 6: Chaining of Decorators

You can also add multiple decorators on a single function. 

When decorators are chained, execution of decorator happens from bottom to top. 

Below `logger` will run first then `timer` will run. 


In [ ]:
@timer(unit="seconds")
@logger("INFO")
def expnonent_sum(end_num: int) -> None:
    total = 0
    for i in range(1, end_num + 1):
        total += i ** i

expnonent_sum(1000)

---
## 🏋️ Day 8 Activities: The Code Upgrader

### Level 1: Add Brackets
Create a decorator called `@add_brackets`. 

When applied to `print_name()`, it should print `[ Tony ]` instead of just `Tony`.


In [ ]:
# Level 1 Code
# Write your decorator here:


# @add_brackets
def print_name():
    print(" Tony ", end="")

### Level 2: The Stopwatch Decorator ⏱️
Get the time execution for any function:
1. Create a decorator `@timer`.
2. Record the start time.
3. Run the original function.
4. Record the end time.
5. Print `"Execution time: [X] seconds"`.
6. Apply it to a function that loops 1 million times.

In [ ]:
# Level 2 Code

### Level 3: The Slow Down Decorator 🐢
1. Create a decorator `@slow_down`.
2. Decorator should make add a delay of 2 seconds to the main fucntion.
3. Apply it to a `countdown(n)` function.

In [ ]:
# Level 3 Code

### Level 4: The Debugger Decorator 🐛
1. Create `@debug`.
2. The wrapper should print exactly what arguments the function was called with.
   *(Hint: Print `args` and `kwargs` inside the wrapper).*
3. Run the main function, save the return value in `result` variable.
4. Print the result.
5. Return the result from decorator.
6. Apply it to `multiply(a, b)`.

In [ ]:
# Level 4 Code

### Level 5: The Resilient Server (`@retry`) (Real Scenario) 📡
**Scenario:** You have a function that connects to a database. Sometimes the internet drops, and it crashes. You want a decorator that automatically tries 3 times before finally crashing.
1. Create a decorator `@retry`.
2. Inside the wrapper, use a `for` loop that runs 3 times.
3. Inside the loop, use `try/except`. 
   * Try to run the function and `return` the result.
   * If it crashes (Exception), print `"Failed, retrying..."`.
4. If the loop finishes all 3 tries and still fails, `raise` the Exception.
5. Apply it to the provided `connect_to_db` function.

In [ ]:
# Level 5 Setup
import random

def connect_to_db():
    if random.choice([True, False, False]): # 66% chance to crash!
        raise ConnectionError("Database timeout!")
    return "Connected successfully!"

# Level 5 Code Here: